# 2. Statistik-Grundlagen in Python

## Real-World-Kontext

Im vorherigen Notebook (`01_python_grundsyntax.ipynb`) hast Du gelernt, wie man Variablen, Listen
und Module benutzt. Jetzt setzen wir das ein, um die **Statistik-Grundlagen**, die Du bereits von
Hand berechnet hast (Mittelwert, Median, Streuung, Korrelation, Regression), in Python nachzubauen.

Wir bleiben beim bekannten Beispiel: sechs gemeldete Kfz-Schadenshöhen, einer davon ein deutlicher
Ausreißer (38.000 EUR), sowie ein zweiter Datensatz zum Zusammenhang zwischen Fahrzeugalter und
Reparaturkosten.

## 🎯 Learning Objectives

By completing this notebook, you will be able to:
- 💡 **Explain** wofür Mittelwert, Median und Modus jeweils stehen und wann sie sich unterscheiden.
- 🛠️ **Apply** `statistics.variance`/`stdev` und die NumPy-Äquivalente korrekt auf ein Datenset.
- 🔍 **Identify** die NumPy-`ddof`-Fallgrube und wissen, wie man sie vermeidet.
- 🛠️ **Apply** `np.corrcoef`, die z-Transformation und `np.polyfit`, um Korrelation und
  Regression in Python zu berechnen.
- 💡 **Explain** wie man R² manuell aus den Vorhersagewerten eines Modells nachrechnet.

## Concept at a Glance

Stell Dir vor, Du hast Deine Statistik-Formeln bisher **mit Taschenrechner und Papier** berechnet.
Das Modul `statistics` ist wie ein Taschenrechner, der bereits alle Formeln eingebaut hat — Du
musst nur die Zahlen hineingeben und die richtige Taste (Funktion) drücken. `numpy` ist ein noch
mächtigerer Taschenrechner, der zusätzlich mit ganzen Zahlenreihen auf einmal rechnen kann.

Unser Running Example bleibt identisch zu den Folien:

```python
schadenshoehen = [800, 1100, 1100, 1400, 2200, 38000]  # EUR, 6 Kfz-Schäden
```

Der letzte Wert (38.000 EUR) ist bewusst ein **Ausreißer** — genau daran zeigen sich die
Unterschiede zwischen den Lagemaßen besonders deutlich.

In [ ]:
# I DO: Lagemaße mit dem statistics-Modul berechnen

import statistics

schadenshoehen = [800, 1100, 1100, 1400, 2200, 38000]

print(f"Mittelwert: {statistics.mean(schadenshoehen):.2f}")
print(f"Median: {statistics.median(schadenshoehen)}")
print(f"Modus: {statistics.mode(schadenshoehen)}")

## Step-by-Step Breakdown — Warum weichen die Lagemaße so stark ab?

- **Mittelwert** (`mean`) ≈ 7433.33 EUR — wird vom Ausreißer (38.000 EUR) stark nach oben gezogen.
- **Median** = 1250.0 EUR — der mittlere Wert der sortierten Liste, unempfindlich gegenüber
  Ausreißern.
- **Modus** = 1100 EUR — der häufigste Wert (kommt zweimal vor).

Das **Skalenproblem** wird hier live im Code sichtbar: Ein einziger extremer Wert kann den
Mittelwert um ein Vielfaches verschieben, während Median und Modus stabil bleiben. Das ist einer
der wichtigsten Gründe, warum Data Scientists nie nur ein einzelnes Lagemaß berichten.

Weiter geht's mit der **Streuung** — wie stark schwanken die Werte um den Mittelwert?

```python
s_var = statistics.variance(schadenshoehen)
s_std = statistics.stdev(schadenshoehen)
```

> 💡 **Good to know:**
> Das `statistics`-Modul nutzt standardmäßig **n−1** im Nenner (die sogenannte Bessel-Korrektur)
> — genau wie Du es vermutlich von Hand für eine **Stichprobe** gerechnet hast.

In [ ]:
# I DO: Varianz und Standardabweichung mit statistics berechnen (nutzt automatisch n-1)

s_var = statistics.variance(schadenshoehen)
s_std = statistics.stdev(schadenshoehen)

print(f"Varianz (statistics): {s_var:.0f}")
print(f"Stdev (statistics): {s_std:.0f}")

## Die NumPy-`ddof`-Fallgrube

> ⚠️ **Common Pitfall:**
> NumPys `np.var()` und `np.std()` nutzen **standardmäßig n** im Nenner (Populationsvarianz),
> nicht n−1! Das ist ein direkter Widerspruch zu `statistics`, das automatisch n−1 nutzt. Wenn Du
> beide Module unreflektiert mischst, bekommst Du **unterschiedliche Zahlen für dieselben Daten** —
> und Deine Streuung wird systematisch unterschätzt.

Die Lösung: NumPy erlaubt Dir, den Freiheitsgrad-Korrekturfaktor explizit über den Parameter
`ddof` (**d**elta **d**egrees **o**f **f**reedom) zu setzen. `ddof=1` erzwingt n−1, also dasselbe
Verhalten wie `statistics`.

```python
np_var_wrong = np.var(schadenshoehen)             # n   - Population, meist falsch für Stichproben
np_var_right = np.var(schadenshoehen, ddof=1)      # n-1 - korrekt für Stichproben
```

**Merksatz:** Für Stichproben-Statistik immer `ddof=1` bei NumPy setzen — sonst unterschätzst Du
die Streuung systematisch.

In [ ]:
# I DO: Die ddof-Fallgrube live sehen

import numpy as np

np_var_wrong = np.var(schadenshoehen)               # ddof fehlt -> nutzt n (falsch für Stichproben!)
np_var_right = np.var(schadenshoehen, ddof=1)        # ddof=1 -> nutzt n-1 (korrekt)

print(f"NumPy (n, falsch):     {np_var_wrong:.0f}")
print(f"NumPy (n-1, korrekt):  {np_var_right:.0f}")
print(f"statistics (n-1):      {s_var:.0f}  <- stimmt mit 'korrekt' überein!")

## Korrelation — Wechsel zum Fahrzeugalter/Reparaturkosten-Datensatz

Für die nächsten Schritte (Korrelation, z-Transformation, Regression) verwenden wir einen zweiten
Running-Example-Datensatz: das Alter eines Fahrzeugs (in Jahren) und die dazugehörigen
Reparaturkosten (in EUR).

```python
alter = np.array([1, 2, 3, 4])           # Jahre
kosten = np.array([200, 300, 500, 600])  # EUR
```

`np.array(...)` erzeugt aus einer Liste ein **NumPy-Array** — eine Datenstruktur, die (anders als
eine normale Python-Liste) mathematische Operationen direkt unterstützt, wie Du gleich siehst.

Die **Korrelationsmatrix** `np.corrcoef(x, y)` liefert eine 2×2-Tabelle. Der Wert oben rechts (oder
unten links — beide sind identisch) ist der Korrelationskoeffizient r.

In [ ]:
# I DO: Korrelationsmatrix berechnen

alter = np.array([1, 2, 3, 4])           # Jahre
kosten = np.array([200, 300, 500, 600])  # EUR

korr_matrix = np.corrcoef(alter, kosten)
print(korr_matrix)

r ≈ 0,99 — genau der Wert, den Du bereits von Hand berechnet hast. Ein starker positiver linearer
Zusammenhang: Mit jedem Lebensjahr steigen die Reparaturkosten im Durchschnitt um rund 140 EUR
(das rechnen wir gleich mit der Regression nach).

## z-Transformation — Schritt für Schritt

Die z-Transformation macht Werte **dimensionslos und vergleichbar**, indem sie in zwei Schritten
rechnet:

**Schritt 1 — Zentrieren:** Ziehe den Mittelwert von jedem Wert ab. Danach hat die neue Variable
selbst einen Mittelwert von genau 0.

```python
alter_zentriert = alter - np.mean(alter)
```

In [ ]:
# I DO: Schritt 1 - Zentrieren

alter_zentriert = alter - np.mean(alter)
print(f"Mittelwert: {np.mean(alter)}")
print(f"Zentriert: {alter_zentriert}")

**Schritt 2 — Skalieren:** Teile jeden zentrierten Wert durch die Standardabweichung
(`ddof=1`, weil wir eine Stichprobe betrachten!). Danach ist jeder Wert ein **z-Wert**: er drückt
aus, wie viele Standardabweichungen dieser Wert vom Mittelwert entfernt liegt.

```python
s = np.std(alter, ddof=1)
alter_z = alter_zentriert / s
```

> ⚠️ **Common Pitfall:**
> Auch hier gilt die ddof-Falle von oben! Vergisst Du `ddof=1`, verschiebt sich der Nenner leicht
> und Deine z-Werte werden minimal falsch berechnet.

In [ ]:
# I DO: Schritt 2 - Skalieren

s = np.std(alter, ddof=1)
alter_z = alter_zentriert / s
print(f"Standardabweichung: {s:.2f}")
print(f"z-Werte: {alter_z.round(2)}")

## Lineare Regression mit `numpy.polyfit()`

Die Regressionsgerade $\hat{Y} = a + bX$ kannst Du von Hand mit den Formeln aus der Vorlesung
berechnen — oder NumPy die Arbeit machen lassen: `np.polyfit(X, Y, deg=1)` findet die
Koeffizienten einer Geraden (Grad 1 = linear), die am besten zu den Punkten passt.

```python
koeffizienten = np.polyfit(alter, kosten, deg=1)
b, a = koeffizienten  # Achtung: höchster Grad zuerst -> Steigung b kommt vor Intercept a!
```

> ⚠️ **Common Pitfall:**
> `np.polyfit` gibt die Koeffizienten in der Reihenfolge **höchster Grad zuerst** zurück. Bei
> Grad 1 heißt das: zuerst die Steigung `b`, dann der Achsenabschnitt (Intercept) `a` — die
> intuitive Reihenfolge `a, b` ist genau andersherum!

In [ ]:
# I DO: Regressionsgerade berechnen

koeffizienten = np.polyfit(alter, kosten, deg=1)
b, a = koeffizienten  # b = Steigung, a = Intercept (Reihenfolge beachten!)

print(f"Steigung b: {b:.1f} EUR/Jahr")
print(f"Intercept a: {a:.1f} EUR")
print(f"Modell: Y = {a:.0f} + {b:.0f} * X")

Das ist exakt dieselbe Regressionsgerade $\hat{Y} = 50 + 140X$, die Du bereits von Hand berechnet
hast — nur dass NumPy sie in einer Zeile findet.

## Modellgüte: R² manuell nachrechnen

R² sagt aus, **wie gut** unser Modell die tatsächlichen Werte erklärt (1.0 = perfekt, 0 = gar
nicht). Die Formel dahinter:

$$R^2 = 1 - \frac{\sum (Y - \hat{Y})^2}{\sum (Y - \bar{Y})^2}$$

Wir setzen sie Schritt für Schritt in Code um:

1. Vorhersagen `y_pred` mit unserem Modell berechnen: $\hat{Y} = a + bX$
2. Die Summe der quadrierten Fehler (Zähler) berechnen
3. Die Summe der quadrierten Abweichungen vom Mittelwert (Nenner) berechnen
4. Alles zu R² zusammensetzen

In [ ]:
# I DO: R² Schritt für Schritt nachrechnen

y_pred = a + b * alter                                            # 1. Vorhersagen
fehler_quadrate = np.sum((kosten - y_pred) ** 2)                  # 2. Zähler
gesamt_streuung = np.sum((kosten - np.mean(kosten)) ** 2)         # 3. Nenner
r_quadrat = 1 - fehler_quadrate / gesamt_streuung                 # 4. R²

print(f"R²: {r_quadrat:.2f}")

Das ist dieselbe R² = 0,98, die Du bereits von Hand berechnet hast. Das Fahrzeugalter erklärt 98 %
der Varianz in den Reparaturkosten — ein fast perfektes lineares Modell für dieses Beispiel.

## Guided Practice — Andere Schwelle, andere Sichtweise

> 🎯 **Your Task:**
> Führe die Zelle unten aus. Sie berechnet dieselben Lagemaße wie oben, aber mit einem leicht
> veränderten Portfolio (ein Schaden wurde nachträglich korrigiert). Beobachte, wie stark sich
> Mittelwert und Median durch die eine geänderte Zahl verschieben — oder eben nicht.

In [ ]:
# WE DO: Leicht verändertes Portfolio - beobachte die Auswirkung auf die Lagemaße

schadenshoehen_neu = [800, 1100, 1100, 1400, 2200, 25000]  # letzter Wert von 38000 auf 25000 korrigiert

print(f"Mittelwert (neu): {statistics.mean(schadenshoehen_neu):.2f}")
print(f"Median (neu): {statistics.median(schadenshoehen_neu)}")
print(f"Stdev (neu, statistics n-1): {statistics.stdev(schadenshoehen_neu):.0f}")

## Mini-Exercise / Self-Check

> 🎯 **Your Task:**
> Ein weiteres Portfolio kommt herein: `[900, 1200, 1200, 1500, 2500, 42000]` EUR. Berechne
> Mittelwert, Median, Varianz und Standardabweichung — **und tappe bewusst in die ddof-Falle**,
> um sie danach selbst zu korrigieren. Vergleiche `np.var(...)` ohne `ddof` mit
> `np.var(..., ddof=1)` und mit `statistics.variance(...)`.

In [ ]:
# =========================================================
# 🎯 EXERCISE: Eigenes Portfolio & die ddof-Falle selbst debuggen
# =========================================================
# Instruction: Ersetze die Platzhalter (???) durch gültigen Code.

portfolio = [900, 1200, 1200, 1500, 2500, 42000]

# 1. Lagemaße mit statistics:
print(f"Mittelwert: {statistics.???(portfolio):.2f}")
print(f"Median: {statistics.???(portfolio)}")

# 2. Varianz mit statistics (nutzt automatisch n-1):
s_var_neu = statistics.???(portfolio)
print(f"Varianz (statistics, n-1): {s_var_neu:.0f}")

# 3. Tappe absichtlich in die ddof-Falle: berechne np.var() OHNE ddof-Parameter
np_var_falsch = np.???(portfolio)
print(f"NumPy Varianz (ohne ddof, also n): {np_var_falsch:.0f}")

# 4. Jetzt korrigiere es: berechne np.var() MIT ddof=1
np_var_richtig = np.var(portfolio, ddof=???)
print(f"NumPy Varianz (mit ddof=1): {np_var_richtig:.0f}")

# 5. Stimmen 'statistics' und 'np.var(..., ddof=1)' jetzt überein?
print(f"Stimmen die Werte überein? {round(s_var_neu) == round(np_var_richtig)}")

## Summary & Key Takeaways

- **Mittelwert, Median, Modus** können bei Ausreißern stark auseinanderlaufen — nie nur ein
  Lagemaß allein berichten.
- `statistics.variance`/`stdev` nutzen automatisch **n−1** (Bessel-Korrektur) — passend für
  Stichproben.
- **NumPys `np.var`/`np.std` nutzen standardmäßig n** — für Stichproben-Statistik **immer**
  `ddof=1` explizit setzen, sonst wird die Streuung systematisch unterschätzt.
- `np.corrcoef(x, y)` liefert die Korrelationsmatrix; der Off-Diagonal-Wert ist r.
- Die z-Transformation läuft immer in zwei Schritten: erst **zentrieren** (Mittelwert abziehen),
  dann **skalieren** (durch Standardabweichung teilen).
- `np.polyfit(x, y, deg=1)` liefert Steigung und Intercept — Achtung, in der Reihenfolge
  **höchster Grad zuerst**.
- R² lässt sich manuell aus den Vorhersagewerten nachrechnen: 1 − (Fehlerquadrate ÷ Gesamtstreuung).

**Weiter geht's:** Bisher haben wir nur *Ergebnisse berechnet und angezeigt*. Im nächsten Notebook
`03_kontrollstrukturen.ipynb` bringst Du Deinem Code bei, selbst zu **entscheiden** — z.B. ob ein
neuer Schaden automatisch als Ausreißer erkannt wird.